# Export a checkpoint to ONNX with an embedded manifest

Produces a `.onnx` file that **describes itself** — the server reads the class
names, input size and normalization straight out of the file, so you register
it and leave the Metadata field empty.

Run this on the **training machine** (the one with the checkpoint and a GPU),
not on the Jetson. Run the cells top to bottom.

Only **Cell 2** needs editing.

## 1 · Dependencies

In [ ]:
%pip install --quiet onnx onnxruntime

import json

import onnx
import onnxruntime as ort
import torch
from torchvision import models

print("torch", torch.__version__, "| onnx", onnx.__version__)

## 2 · Configure — the only cell you edit

Three things must match your **training/eval transform exactly**. A mismatch
here doesn't error, it silently corrupts every prediction:

- `CLASSES` — the order training used for label indices. `ImageFolder` sorts
  alphabetically; if you set the order by hand, use that order.
- `IMAGE_SIZE` — the input size the model was trained at.
- `MEAN` / `STD` — the normalization constants from your eval transform.

In [ ]:
CHECKPOINT = r"D:\GibsonGuitar\CodeMadeByQuin\quin\resnet_50fold_3.pth"
OUT = "gibson_resnet50_fold3.onnx"

CLASSES = ["2A", "3A", "4A"]        # order = label index order = ordinal order
IMAGE_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

MODEL_ID = "gibson-resnet50"
VERSION = "fold3-v1"
VAL_METRICS = {"accuracy": 0.3893, "macro_f1": 0.3701}

## 3 · Rebuild the architecture and load the weights

The asserts matter. `strict=False` would otherwise silently skip a mismatched
classifier head and you'd export a **randomly initialised** one — a model that
loads fine and predicts noise.

In [ ]:
model = models.resnet50(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, len(CLASSES))

ckpt = torch.load(CHECKPOINT, map_location="cpu")
# Checkpoints are often wrapped in a dict; take the state dict whichever way.
state = ckpt.get("model_state_dict") or ckpt.get("state_dict") or ckpt
# DataParallel prefixes every key with "module."
state = {k.replace("module.", ""): v for k, v in state.items()}

missing, unexpected = model.load_state_dict(state, strict=False)
assert not missing, f"checkpoint is missing weights for: {missing}"
assert not unexpected, f"checkpoint has unexpected keys: {unexpected}"

model.eval()
print(f"loaded {CHECKPOINT}")
print(f"head: {model.fc}")

### 3b · Sanity check (optional but recommended)

Confirms the loaded model isn't degenerate before you ship it. If every random
input yields the same class, the head didn't load.

In [ ]:
with torch.no_grad():
    probe = model(torch.randn(8, 3, IMAGE_SIZE, IMAGE_SIZE))
    preds = probe.argmax(1).tolist()

print("logit spread :", round(probe.std().item(), 4))
print("predictions  :", [CLASSES[i] for i in preds])
if len(set(preds)) == 1:
    print("\n!! every input gave the same class — suspect an unloaded head")

## 4 · Export to ONNX

In [ ]:
torch.onnx.export(
    model,
    torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE),
    OUT,
    input_names=["images"],
    output_names=["logits"],
    # Batch stays dynamic so the server can send several crops at once.
    dynamic_axes={"images": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)
print(f"exported {OUT}")

## 5 · Stamp the manifest into the file

`metadata_props` is ONNX's equivalent of TorchScript's `_extra_files`. The
server looks for the key `metadata.json`.

In [ ]:
meta = {
    "classes": CLASSES,
    "task": "classification",
    "image_size": IMAGE_SIZE,
    "max_views": 1,
    "patch_mode": False,
    "normalize_mean": MEAN,
    "normalize_std": STD,
    "temperature": 1.0,          # >1.0 if you fitted a calibration scalar
    "variant": VERSION,
    "val_metrics": VAL_METRICS,
}

m = onnx.load(OUT)
# Replace rather than append, so re-running doesn't leave duplicate keys.
keep = [p for p in m.metadata_props if p.key not in ("metadata.json", "metadata")]
del m.metadata_props[:]
m.metadata_props.extend(keep)
entry = m.metadata_props.add()
entry.key = "metadata.json"
entry.value = json.dumps(meta)
onnx.save(m, OUT)

print(json.dumps(meta, indent=2))

## 6 · Verify it round-trips

Reads the file back exactly the way the server will.

In [ ]:
sess = ort.InferenceSession(OUT, providers=["CPUExecutionProvider"])

embedded = sess.get_modelmeta().custom_metadata_map.get("metadata.json")
assert embedded, "no embedded metadata — the server would reject this file"
assert json.loads(embedded)["classes"] == CLASSES

logits = sess.run(None, {"images": torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).numpy()})[0]
assert logits.shape == (1, len(CLASSES)), logits.shape

print("embedded metadata :", embedded[:120], "...")
print("output shape      :", logits.shape)
print(f"\nOK — {OUT} is ready to register.")

## 7 · Register it

Copy the `.onnx` to the Jetson, or upload through the web UI:

**Models → Register an archive** → pick the file, set the id and version,
tick **Activate**, leave **Metadata JSON** empty.

Or from a terminal:

```bash
curl -X POST http://JETSON-IP:8000/v1/models \
  -H "X-API-Key: $ADMIN_KEY" \
  -F "archive=@gibson_resnet50_fold3.onnx" \
  -F "model_id=gibson-resnet50" \
  -F "version=fold3-v1" \
  -F "activate=true"
```

After loading it, check the **Performance** page: if the provider under the GPU
gauge reads `CPUExecutionProvider`, ONNX is running on CPU (the PyPI aarch64
wheel is CPU-only) — see the README for the GPU wheel.